# 03 — Baseline Models: Ridge & Logistic Regression

**Purpose:** Establish the linear baseline IC before introducing tree-based complexity.

**The right interpretation of this notebook:**
If XGBoost (notebook 04) does not clearly outperform Ridge regression here, it means:
- The non-linear interactions in the feature set are weak, OR
- The features are already well-specified for a linear model

Either way, that is a valuable finding. A simpler model that performs comparably is always preferable.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src.models import BaselineRankModel, BaselineClassifier, WalkForwardRunner
from src.targets import align_features_targets
from src.metrics import rolling_ic, rolling_topk_spread, ic_summary, hit_rate
from src.plotting import plot_rolling_ic, plot_topk_spread, plot_quintile_returns

PROCESSED = cfg.processed_dir()
print('OK')

In [ ]:
features = pd.read_parquet(PROCESSED / 'features.parquet')
targets  = pd.read_parquet(PROCESSED / 'targets.parquet')
X, y_all = align_features_targets(features, targets)
y_rank, y_binary, y_ret = y_all['rank_target'], y_all['top_label'], y_all['fwd_return']
print(f'Dataset: {X.shape}')

## 1. Ridge Regression — Walk-Forward

In [ ]:
from src.validation import WalkForwardSplitter

splitter = WalkForwardSplitter(
    min_train_months=cfg.get('validation.min_train_months', 24),
    test_months=cfg.get('validation.test_months', 3),
    gap_days=cfg.get('validation.gap_days', 5),
)

runner = WalkForwardRunner(X, y_rank, y_binary=y_binary, splitter=splitter)

ridge_results = runner.run_model(BaselineRankModel, target='rank')
print('\nRidge summary:', ridge_results.summary())

In [ ]:
# Plot IC time series
ridge_scores  = ridge_results.all_oos_scores
ridge_returns = ridge_results.all_oos_returns
ridge_ic      = rolling_ic(ridge_scores, ridge_returns)

fig = plot_rolling_ic(ridge_ic, model_name='Ridge (Baseline)')
plt.show()

In [ ]:
# Top-K spread
ridge_spread = rolling_topk_spread(ridge_scores, ridge_returns)
print(f'Ridge hit rate: {hit_rate(ridge_spread):.1%}')

fig = plot_topk_spread(ridge_spread, model_name='Ridge (Baseline)')
plt.show()

## 2. Logistic Regression (Top Quintile Classification)

In [ ]:
logistic_results = runner.run_model(BaselineClassifier, target='binary')
print('Logistic summary:', logistic_results.summary())

logit_ic = rolling_ic(logistic_results.all_oos_scores, logistic_results.all_oos_returns)
fig = plot_rolling_ic(logit_ic, model_name='Logistic (Baseline)')
plt.show()

## 3. Baseline Comparison

In [ ]:
from src.plotting import plot_ic_comparison

ic_dict = {
    'Ridge':    rolling_ic(ridge_results.all_oos_scores, ridge_results.all_oos_returns),
    'Logistic': rolling_ic(logistic_results.all_oos_scores, logistic_results.all_oos_returns),
}

fig = plot_ic_comparison(ic_dict)
plt.show()

print('\nBaseline IC summary:')
for name, ic_s in ic_dict.items():
    s = ic_summary(ic_s)
    print(f'  {name:<12}: IC={s["mean_ic"]:+.4f}  ICIR={s["icir"]:+.3f}  hit={s["pct_positive"]:.1%}')

## Summary
Fill in after running:
- Ridge IC: X.XXXX | ICIR: X.XXX
- Logistic IC: X.XXXX | ICIR: X.XXX
- Observations: (which model is better? are they similar?)

These numbers are the **floor** for tree-based models in notebook 04.

→ Proceed to `04_tree_models.ipynb`